# Day 74: Recommendation Systems Basics

## Setting Up Ratings Data
Recommendation systems (used everywhere from Netflix to Amazon to Spotify) typically start from a user-item matrix: rows are users, columns are items (movies here), and each cell holds how that user rated that item. This matrix is almost always sparse in real applications - most users only rate a small fraction of all items - and here we use 0 as a stand-in for "not rated yet," which is a simplification worth remembering (a real system distinguishes "unrated" from "rated 0/poorly," often using NaN or a separate mask, since treating unrated as a real low rating can bias similarity calculations). This small 5-user, 4-movie matrix is intentionally tiny so we can compute and inspect similarities by hand/eye in the next steps, but the same logic (just at massive scale, with smarter handling of sparsity) underlies real production recommender systems.</cell id="cell-1">


In [ ]:
import pandas as pd
import numpy as np

ratings = pd.DataFrame({
    'Inception': [5, 4, 0, 0, 3],
    'Titanic': [3, 0, 4, 5, 0],
    'Avengers': [4, 5, 0, 3, 4],
    'Notebook': [0, 3, 5, 4, 0]
}, index=['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'])  # rows = users, columns = movies, values = ratings (0 = not rated)
print(ratings)

## Item-Based Similarity
Item-based collaborative filtering asks: "which movies got rated similarly by the same users?" - if two movies tend to receive similar ratings across users, they're considered similar, and that similarity can be used to recommend one based on the other. Cosine similarity measures how similar two vectors are by the angle between them (1 = identical direction, 0 = unrelated, -1 = opposite), which works well here because it's insensitive to a rating vector's overall magnitude and focuses on the pattern of ratings. `ratings.T` transposes the matrix so each row becomes a movie's ratings across all users (instead of a user's ratings across all movies), and `cosine_similarity` then compares every pair of these movie vectors, producing a movie x movie similarity matrix - a value close to 1 means two movies were liked/disliked by similar sets of users.</cell id="cell-3">


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = pd.DataFrame(
    cosine_similarity(ratings.T), index=ratings.columns, columns=ratings.columns  # transpose so rows = movies, then compare movie rating patterns
)
print(item_similarity.round(2))

## Making a Recommendation
Once we have the item-similarity matrix, turning it into an actual recommendation is simple: pick the column for a movie a user already liked ('Inception'), sort it in descending order, and read off the top matches - the highest values (excluding Inception's similarity with itself, which is always 1.0) are the movies most similar in rating pattern, and are good candidates to recommend to someone who liked Inception. In a complete recommender you'd also filter out movies the user has already rated before presenting recommendations; this cell only shows the ranking step. This same "sort the similarity column" pattern is the core mechanic behind "customers who liked X also liked Y" style recommendations.</cell id="cell-5">


In [ ]:
similar_to_inception = item_similarity['Inception'].sort_values(ascending=False)  # movies most similar to Inception, most similar first
print(similar_to_inception)

## User-Based Collaborative Filtering
User-based collaborative filtering flips the question around: instead of "which movies are similar," it asks "which users have similar taste?" - if two users tend to rate movies the same way, we can recommend one user's favorite movies to the other, on the assumption that similar people will like similar things. The computation is exactly the same cosine similarity calculation as before, just applied to the original (non-transposed) ratings matrix, so each row (a user's rating vector across all movies) is compared to every other user's row, producing a user x user similarity matrix. In practice, item-based filtering tends to be more stable than user-based filtering in large systems, because item similarity patterns change more slowly than the huge and constantly-shifting space of individual user preferences - but both are foundational techniques worth knowing.</cell id="cell-7">


In [ ]:
user_similarity = pd.DataFrame(
    cosine_similarity(ratings), index=ratings.index, columns=ratings.index  # compare each user's rating pattern to every other user's
)
print(user_similarity.round(2))